In [1]:
"""
=============================================================
  IP Malicious Labeler using AlienVault OTX API
=============================================================
  Purpose:
    Load a dataset (local CSV path or URL), look up each IP
    against OTX, and fill ONLY the High_Incident_Label column:
      1 = MALICIOUS  (appears in OTX pulses OR negative reputation)
      0 = CLEAN      (no threat indicators found)
     -1 = ERROR      (API failed after retries — review manually)

    All other columns in your dataset are left untouched.

  Usage:
    python ip_malicious_labeler.py

  Requirements:
    pip install requests pandas
=============================================================
"""

import sys
import time
import requests
import pandas as pd
from typing import Optional
from datetime import datetime

# ─────────────────────────────────────────────
#  CONFIG
# ─────────────────────────────────────────────
OTX_API_KEY      = "dac97896c6af5191c5cc3b367e104d98bb813165721a4b2ac65488b7b5515ed1"  # ← replace or enter at runtime
OTX_BASE_URL     = "https://otx.alienvault.com/api/v1/indicators/IPv4"
RATE_LIMIT_DELAY = 0.2   # seconds between API calls
MAX_RETRIES      = 5     # retries on timeout/connection error
RETRY_DELAY      = 2.0   # seconds to wait between retries

# ─────────────────────────────────────────────────────────────
#  IP Validation
# ─────────────────────────────────────────────────────────────
def validate_ip(ip: str) -> bool:
    """Returns True only for valid IPv4 strings (not 'nan', not empty)."""
    ip = str(ip).strip().lower()
    if ip in ("nan", "", "none"):
        return False
    parts = ip.split(".")
    if len(parts) != 4:
        return False
    for part in parts:
        if not part.isdigit():
            return False
        if not (0 <= int(part) <= 255):
            return False
    return True

# ─────────────────────────────────────────────────────────────
#  OTX API - Fetch with retry
# ─────────────────────────────────────────────────────────────
def fetch_otx_general(ip: str, api_key: str) -> Optional[dict]:
    """
    Fetch OTX /general endpoint for an IP.
    Retries up to MAX_RETRIES times on timeout or connection error.
    Returns None if all attempts fail.
    """
    url     = f"{OTX_BASE_URL}/{ip}/general"
    headers = {"X-OTX-API-KEY": api_key}

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = requests.get(url, headers=headers, timeout=15)
            resp.raise_for_status()
            return resp.json()

        except requests.exceptions.HTTPError as e:
            # 4xx errors won't improve on retry
            print(f"    [HTTP {resp.status_code}] {e}")
            return None

        except (requests.exceptions.ConnectionError,
                requests.exceptions.Timeout) as e:
            print(f"    [Attempt {attempt}/{MAX_RETRIES}] {type(e).__name__} — "
                  f"{'retrying...' if attempt < MAX_RETRIES else 'giving up.'}")
            if attempt < MAX_RETRIES:
                time.sleep(RETRY_DELAY)

    return None

# ─────────────────────────────────────────────────────────────
#  LABEL LOGIC  →  returns 1, 0, or -1
# ─────────────────────────────────────────────────────────────
def get_label(ip: str, api_key: str) -> int:
    """
    Returns:
      1   if pulse_count > 0  OR  reputation < 0  → MALICIOUS
      0   if OTX returned data but no threat indicators → CLEAN
     -1   if OTX API failed after all retries       → ERROR (review manually)
    """
    data = fetch_otx_general(ip, api_key)

    if data is None:
        return -1   # API failure — do not assume clean

    pulse_count = data.get("pulse_info", {}).get("count", 0) or 0

    raw_rep    = data.get("reputation")
    reputation = int(raw_rep) if raw_rep is not None else 0

    return 1 if (pulse_count > 0 or reputation < 0) else 0

# ─────────────────────────────────────────────────────────────
#  DATASET LOADER
# ─────────────────────────────────────────────────────────────
def load_dataset(source: str) -> pd.DataFrame:
    source = source.strip().strip('"').strip("'")
    try:
        df = pd.read_csv(source)
        print(f"  Loaded {len(df)} row(s) from: {source}")
        return df
    except Exception as e:
        print(f"  [Error] Could not load dataset: {e}")
        return pd.DataFrame()

def detect_ip_column(df: pd.DataFrame, hint: str = "") -> Optional[str]:
    if hint and hint in df.columns:
        return hint
    candidates = [c for c in df.columns if "ip" in c.lower()]
    if candidates:
        print(f"  [Auto-detect] Using '{candidates[0]}' as IP column.")
        return candidates[0]
    return None

# ─────────────────────────────────────────────────────────────
#  CORE - Fill High_Incident_Label column only
# ─────────────────────────────────────────────────────────────
def label_dataset(df: pd.DataFrame, ip_column: str, api_key: str) -> pd.DataFrame:
    all_ips  = df[ip_column].astype(str).str.strip().tolist()
    valid    = [ip for ip in dict.fromkeys(all_ips) if validate_ip(ip)]
    invalid  = [ip for ip in dict.fromkeys(all_ips) if not validate_ip(ip)]
    total    = len(valid)

    if invalid:
        print(f"  [Skipped] {len(invalid)} invalid/empty IP(s): {invalid[:5]}"
              f"{'...' if len(invalid) > 5 else ''}")

    print(f"\n  Found {total} unique valid IP(s) in '{ip_column}'.")
    print(f"  Querying OTX...\n")
    print(f"  {'#':<6} {'IP':<18} {'Result'}")
    print(f"  {'-'*6} {'-'*18} {'-'*22}")

    label_map = {}

    for i, ip in enumerate(valid, 1):
        label         = get_label(ip, api_key)
        label_map[ip] = label

        if label == 1:
            status = "1 - MALICIOUS"
        elif label == 0:
            status = "0 - CLEAN"
        else:
            status = "-1 - API ERROR (review)"

        print(f"  {i:<6} {ip:<18} {status}")
        time.sleep(RATE_LIMIT_DELAY)

    print()

    # Map labels back; IPs that failed validation stay as -1
    df_out = df.copy()
    df_out["High_Incident_Label"] = (
        df_out[ip_column]
        .astype(str).str.strip()
        .map(lambda ip: label_map.get(ip, -1) if validate_ip(ip) else -1)
        .astype(int)
    )

    return df_out

# ─────────────────────────────────────────────────────────────
#  SUMMARY
# ─────────────────────────────────────────────────────────────
def print_summary(df: pd.DataFrame) -> None:
    col   = df["High_Incident_Label"]
    total = len(df)

    if total == 0:
        print("  No rows to summarize.")
        return

    malicious = int((col == 1).sum())
    clean     = int((col == 0).sum())
    errors    = int((col == -1).sum())

    print(f"\n{'=' * 48}")
    print(f"  LABELING SUMMARY")
    print(f"{'=' * 48}")
    print(f"  Total rows            : {total}")
    print(f"  MALICIOUS  (label= 1) : {malicious:<6} ({100*malicious/total:.1f}%)")
    print(f"  CLEAN      (label= 0) : {clean:<6} ({100*clean/total:.1f}%)")
    print(f"  API ERROR  (label=-1) : {errors:<6} ({100*errors/total:.1f}%)  ← review these")
    print(f"{'=' * 48}\n")

# ─────────────────────────────────────────────────────────────
#  MAIN
# ─────────────────────────────────────────────────────────────
def main():
    print("=" * 55)
    print("  IP Malicious Labeler  —  OTX Edition")
    print("=" * 55)

    # Dataset source
    print("\n  Paste your dataset path or URL.")
    print("  Examples:")
    print("    C:/Users/you/data/traffic.csv")
    print("    /home/user/data/ips.csv")
    print("    https://raw.githubusercontent.com/.../data.csv\n")
    source = input("  Dataset path / URL: ").strip()

    df = load_dataset(source)
    if df.empty:
        print("  Aborting — could not load dataset.")
        sys.exit(1)

    print(f"\n  Columns found: {list(df.columns)}")

    # IP column
    col_hint  = input("\n  IP column name (Enter to auto-detect): ").strip()
    ip_column = detect_ip_column(df, col_hint)
    if not ip_column:
        print(f"  [Error] No IP column found. Available: {list(df.columns)}")
        sys.exit(1)

    # OTX API key
    entered = input("\n  OTX API Key (Enter to use config value): ").strip()
    api_key = entered if entered else OTX_API_KEY
    if api_key == "YOUR_OTX_API_KEY_HERE":
        print("\n  WARNING: No API key set — all labels will be -1 (error).")
        print("  Get a free key at: https://otx.alienvault.com\n")

    # Process
    df_out = label_dataset(df, ip_column, api_key)
    print_summary(df_out)

    # Save
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    out_path  = f"labeled_dataset_{timestamp}.csv"
    df_out.to_csv(out_path, index=False)
    print(f"  Saved to: {out_path}")

    # Preview
    print(f"\n  Preview (first 5 rows):")
    print(df_out[[ip_column, "High_Incident_Label"]].head().to_string(index=False))
    print()

if __name__ == "__main__":
    main()


  IP Malicious Labeler  —  OTX Edition

  Paste your dataset path or URL.
  Examples:
    C:/Users/you/data/traffic.csv
    /home/user/data/ips.csv
    https://raw.githubusercontent.com/.../data.csv

  Loaded 10000 row(s) from: C:/Users/Leo/Desktop/Trabaho/dataset 11-20/final datasets_part012.csv

  Columns found: ['src_ip', 'listsource', 'listrecency', 'sourcecount', 'firstseen', 'lastseen', 'DaysElapsed', 'Incidents', 'Episode', 'UniqueDstIPCount', 'UniqueDstPortCount', 'PredIncidents', 'count_unique', 'IP_Length', 'Mean_Octet_Value', 'Std_Octet_Value', 'Max_Octet_Value', 'Min_Octet_Value', 'Sum_Octets', 'Repetition_Ratio', 'Numeric_Entropy', 'High_Incident_Label']

  Found 10000 unique valid IP(s) in 'src_ip'.
  Querying OTX...

  #      IP                 Result
  ------ ------------------ ----------------------
  1      113.221.46.58      1 - MALICIOUS
  2      113.221.46.59      1 - MALICIOUS
  3      113.221.46.6       1 - MALICIOUS
  4      113.221.46.61      1 - MALICIOUS
  5 